# 02 · SQL, RFM and evaluated entity resolution

Question: how do repeat purchasing, concentration and source identity interact? This notebook keeps operational masters, raw departmental records and evaluation labels separate.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd  # noqa: E402 — repository source bootstrap above
from intelligent_mdm.ingestion import generate, validate_bundle  # noqa: E402
bundle = generate("demo", seed=42)
assert validate_bundle(bundle).empty
bundle.metadata

{'company': 'Lumen Distribution',
 'size': 'demo',
 'seed': 42,
 'as_of': '2026-09-30',
 'currency': 'EUR',
 'period_start': '2024-10-01',
 'period_end': '2026-09-30',
 'generator_version': 1,
 'synthetic': True}

In [2]:
from intelligent_mdm.sql import query_text, run_named
print(query_text("monthly_sales"))
monthly = run_named(bundle, "monthly_sales")
assert abs(monthly.revenue.sum() - bundle.tables["sales"].amount.sum()) < .01
monthly

-- CTE + temporal aggregation + window function. Zero-sales months are included.
WITH calendar AS (
 SELECT unnest(generate_series(date_trunc('month', min(order_date)),
                               date_trunc('month', max(order_date)), interval '1 month')) AS month
 FROM orders
), monthly AS (
 SELECT date_trunc('month', sale_date) AS month, sum(amount) AS revenue,
        count(DISTINCT order_id) AS orders
 FROM sales GROUP BY 1
), filled AS (
 SELECT c.month, coalesce(m.revenue, 0) AS revenue, coalesce(m.orders, 0) AS orders
 FROM calendar c LEFT JOIN monthly m USING (month)
), prior AS (
 SELECT *, lag(revenue) OVER (ORDER BY month) AS previous_revenue FROM filled
)
SELECT *, 100 * (revenue-previous_revenue) / nullif(previous_revenue, 0) AS mom_pct
FROM prior ORDER BY month



,month,revenue,orders,previous_revenue,mom_pct
0,2024-10-01,38654.89,58,NaN,NaN
1,2024-11-01,72708.46,62,38654.89,88.096409
2,2024-12-01,78164.35,62,72708.46,7.503790
3,2025-01-01,57203.64,47,78164.35,-26.816202
4,2025-02-01,35972.70,57,57203.64,-37.114666
5,2025-03-01,36333.51,47,35972.70,1.003011
6,2025-04-01,56138.87,62,36333.51,54.509900
7,2025-05-01,44412.95,63,56138.87,-20.887346
8,2025-06-01,71181.92,58,44412.95,60.272893
9,2025-07-01,40232.11,54,71181.92,-43.479875


In [3]:
from intelligent_mdm.analytics import business_summary, rfm
business_summary(bundle)

{'revenue': 1192155.2799999998,
 'orders': 1321,
 'customers': 120,
 'active_customers': 78,
 'inactive_customers': 42,
 'average_order': 902.4642543527629,
 'top20_share': 0.6988348447359981,
 'active_days': 180,
 'as_of': '2026-09-30'}

In [4]:
segments = rfm(bundle)
segments.groupby("segment").agg(customers=("customer_id", "size"), frequency=("frequency", "median"), revenue=("monetary", "sum"))

,customers,frequency,revenue
segment,,,
At risk,21,1.0,28727.11
Champions,37,7.0,442766.97
Loyal,14,4.0,57722.18
New,2,1.0,2163.11
No purchases,21,0.0,0.00
Occasional,25,2.0,37768.40


## Interpret unsupervised clusters

Log-scaled R/F/M measurements are standardized. K is selected by in-sample silhouette; this measures internal separation, not generalization. Compare the cluster profiles in original units with rule-based RFM segments.

In [5]:
from intelligent_mdm.machine_learning import cluster_customers, detect_anomalies
clusters = cluster_customers(segments)
clusters["scores"]

,k,silhouette,inertia
0,2,0.438838,148.687095
1,3,0.360580,100.261010
2,4,0.398545,72.384832
3,5,0.342509,58.799816
4,6,0.383023,48.829422


In [6]:
clusters["profiles"]

,customers,recency,frequency,monetary
cluster,,,,
0,30,24.533333,15.866667,14386.328333
1,69,131.391304,2.463768,1993.593043


In [7]:
clusters["rfm_comparison"]

segment,At risk,Champions,Loyal,New,Occasional
cluster,,,,,
0,0,28,2,0,0
1,21,9,12,2,25


## Compare matching methods with the full labelled-pair denominator

Precision/recall/F1 exclude conflicted proposals and include missed candidate pairs. The base generator contains easy numbered company names. A separate challenge includes spelling variants, contradictory identities, and aliases with no shared tokens.

In [8]:
from intelligent_mdm.matching import match, evaluate
from intelligent_mdm.reporting.experiments import challenge
source, labels = challenge()
pd.DataFrame([{"method": method, **evaluate(match(source, method), labels, 85.)} for method in ["exact", "fuzzy"]])

,method,threshold,tp,fp,fn,precision,recall,f1,labelled_positive_pairs,predicted_pairs,seconds,candidate_pairs,excluded_common_tokens
0,exact,85.0,2,0,3,1.0,0.4,0.571429,5,2,0.015037,4,0
1,fuzzy,85.0,3,0,2,1.0,0.6,0.750000,5,3,0.012248,6,0


## Anomaly detection is not error detection

The contamination parameter fixes a review workload. Multivariate Isolation Forest and univariate IQR/z-score answer different questions. Legitimate bulk buying can be flagged.

In [9]:
lines = bundle.tables["order_lines"]
forest = detect_anomalies(lines, ["quantity", "unit_price", "net_amount"])
lines.join(forest).sort_values("anomaly_score", ascending=False).head(10)

,line_id,order_id,product_id,quantity,unit_price,unit_cost,discount,net_amount,eligible,anomaly,anomaly_score
2285,L00002285,O0000776,P0043,200,137.93,76.54,0.00,27586.00,True,True,0.841533
1045,L00001045,O0000366,P0000,125,106.79,71.48,0.00,13348.75,True,True,0.825883
2317,L00002317,O0000786,P0022,200,69.97,41.06,0.10,12594.60,True,True,0.825527
545,L00000545,O0000199,P0010,100,105.84,75.07,0.00,10584.00,True,True,0.823093
3878,L00003878,O0001301,P0025,175,30.73,19.91,0.00,5377.75,True,True,0.822303
3876,L00003876,O0001301,P0005,150,68.20,53.61,0.00,10230.00,True,True,0.819892
1334,L00001334,O0000464,P0030,125,74.53,56.15,0.05,8850.44,True,True,0.819499
1101,L00001101,O0000383,P0008,125,79.70,60.25,0.00,9962.50,True,True,0.818711
191,L00000191,O0000067,P0007,225,14.89,9.37,0.05,3182.74,True,True,0.818602
528,L00000528,O0000192,P0040,100,49.26,25.47,0.05,4679.70,True,True,0.813541


## Executed multi-scale experiments

The files below were produced by the CLI using both dataset sizes and seeds 42/123. Timings are measurements from this machine, not service-level guarantees. Semantic matching remains NOT RUN without an explicitly supplied local model.

In [10]:
results = ROOT / "docs" / "experiments"
pd.read_csv(results / "performance.csv")

,size,seed,generation_seconds,rows,dataframe_bytes,eda_seconds,monthly_sql_seconds,anomaly_comparison_seconds
0,demo,42,0.107063,11622,2521397,0.028141,0.113525,0.432547
1,demo,123,0.105503,11542,2502716,0.030196,0.106752,0.483372
2,large,42,1.151449,194025,41678888,0.152105,0.143341,2.084949
3,large,123,1.062602,194363,41754994,0.155237,0.155117,2.522751


In [11]:
pd.read_csv(results / "anomalies.csv")

,size,seed,method,flags,bulk_labels,precision,recall,f1,target
0,demo,42,IQR_quantity,99,21,0.212121,1.000000,0.350000,Constructed legitimate bulk purchases; NOT err...
1,demo,42,Z_quantity,20,21,1.000000,0.952381,0.975610,Constructed legitimate bulk purchases; NOT err...
2,demo,42,IsolationForest,126,21,0.166667,1.000000,0.285714,Constructed legitimate bulk purchases; NOT err...
3,demo,123,IQR_quantity,94,20,0.212766,1.000000,0.350877,Constructed legitimate bulk purchases; NOT err...
4,demo,123,Z_quantity,19,20,1.000000,0.950000,0.974359,Constructed legitimate bulk purchases; NOT err...
5,demo,123,IsolationForest,126,20,0.158730,1.000000,0.273973,Constructed legitimate bulk purchases; NOT err...
6,large,42,IQR_quantity,1817,359,0.197578,1.000000,0.329963,Constructed legitimate bulk purchases; NOT err...
7,large,42,Z_quantity,320,359,1.000000,0.891365,0.942563,Constructed legitimate bulk purchases; NOT err...
8,large,42,IsolationForest,2159,359,0.166281,1.000000,0.285147,Constructed legitimate bulk purchases; NOT err...
9,large,123,IQR_quantity,1885,360,0.190981,1.000000,0.320713,Constructed legitimate bulk purchases; NOT err...


In [12]:
import json
json.loads((results / "summary.json").read_text())["semantic"]

'NOT RUN — no local embedding model provided'

## Decisions and next experiments

Validate real-world labels and no-token aliases before using similarity thresholds operationally. Review individual cluster profiles rather than interpreting cluster numbers as ranks. Two constructed seasonal cycles do not justify a forecasting claim. No source records were merged or deleted during this notebook.